<a href="https://colab.research.google.com/github/codebyBP/build-your-own-x/blob/master/Beginner_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Functional data Pipeline

## 1.1 Character Tokenizer

In [3]:
# =====================================================================
# 1. DOWNLOAD THE DATASET
# =====================================================================
# We will use a small dataset: Tiny Shakespeare.
# It contains all of Shakespeare's plays and is perfect for a small LLM.
import urllib.request

url = "https://huggingface.co/datasets/Trelis/tiny-shakespeare/resolve/main/input.txt"
filename = "shakespeare.txt"
urllib.request.urlretrieve(url, filename)

# Read the file to see what the text looks like
with open(filename, 'r', encoding='utf-8') as f:
    text = f.read()

print(f"Total characters in dataset: {len(text)}")
print("--- Sample Text ---")
print(text[:150])
print("-------------------\n")

Total characters in dataset: 1115394
--- Sample Text ---
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

A
-------------------



In [4]:
# =====================================================================
# 2. CREATE THE VOCABULARY
# =====================================================================
chars = sorted(list(set(text)))
vocab_size = len(chars)

print(f"Vocabulary Size (Unique Characters): {vocab_size}")
print(f"All unique characters: {''.join(chars)}\n")


Vocabulary Size (Unique Characters): 65
All unique characters: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz



In [7]:
# =====================================================================
# 3. BUILD THE TOKENIZER DICTIONARIES
# =====================================================================
char_to_int = { ch:i for i,ch in enumerate(chars) }
int_to_char = { i:ch for i,ch in enumerate(chars) }

def encode(s):
    return [char_to_int[c] for c in s]

def decode(l):
    return ''.join([int_to_char[i] for i in l])

print("Dictionary built")
print ("char to int is")
print (char_to_int)
print ("int to char is")
print (int_to_char)

Dictionary built
char to int is
{'\n': 0, ' ': 1, '!': 2, '$': 3, '&': 4, "'": 5, ',': 6, '-': 7, '.': 8, '3': 9, ':': 10, ';': 11, '?': 12, 'A': 13, 'B': 14, 'C': 15, 'D': 16, 'E': 17, 'F': 18, 'G': 19, 'H': 20, 'I': 21, 'J': 22, 'K': 23, 'L': 24, 'M': 25, 'N': 26, 'O': 27, 'P': 28, 'Q': 29, 'R': 30, 'S': 31, 'T': 32, 'U': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 'i': 47, 'j': 48, 'k': 49, 'l': 50, 'm': 51, 'n': 52, 'o': 53, 'p': 54, 'q': 55, 'r': 56, 's': 57, 't': 58, 'u': 59, 'v': 60, 'w': 61, 'x': 62, 'y': 63, 'z': 64}
int to char is
{0: '\n', 1: ' ', 2: '!', 3: '$', 4: '&', 5: "'", 6: ',', 7: '-', 8: '.', 9: '3', 10: ':', 11: ';', 12: '?', 13: 'A', 14: 'B', 15: 'C', 16: 'D', 17: 'E', 18: 'F', 19: 'G', 20: 'H', 21: 'I', 22: 'J', 23: 'K', 24: 'L', 25: 'M', 26: 'N', 27: 'O', 28: 'P', 29: 'Q', 30: 'R', 31: 'S', 32: 'T', 33: 'U', 34: 'V', 35: 'W', 36: 'X', 37: 'Y', 38: 'Z', 39: 'a', 40: 'b', 41: 'c', 42: 'd

In [8]:
# =====================================================================
# 4. TEST YOUR CUSTOM TOKENIZER
# =====================================================================
test_sentence = "hello world!"
encoded_ids = encode(test_sentence)
decoded_text = decode(encoded_ids)

print("--- Tokenizer Test ---")
print(f"Original Text: '{test_sentence}'")
print(f"Encoded Integer IDs: {encoded_ids}")
print(f"Decoded Text Back:   '{decoded_text}'")

--- Tokenizer Test ---
Original Text: 'hello world!'
Encoded Integer IDs: [46, 43, 50, 50, 53, 1, 61, 53, 56, 50, 42, 2]
Decoded Text Back:   'hello world!'


## 1.2 Prepare data for the tansformer

In [9]:
import torch

# Encode the entire Shakespeare dataset into a single massive list of integers
data = torch.tensor(encode(text), dtype=torch.long)

# Look at the shape of the data and the first 20 tokens
print(f"Data Shape: {data.shape}")
print(f"First 20 tokens: {data[:20]}")

# Split the data into Training and Validation sets
# 90% of the text will be used to train the model, 10% to test it
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f"Training tokens: {len(train_data)}")
print(f"Validation tokens: {len(val_data)}")


Data Shape: torch.Size([1115394])
First 20 tokens: tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56])
Training tokens: 1003854
Validation tokens: 111540
